In [37]:
import os
import json
import pandas as pd
import traceback

In [38]:
from langchain.chat_models import ChatOpenAI

In [39]:
from dotenv import load_dotenv
load_dotenv()

True

In [40]:
KEY=os.getenv("my_key")

In [41]:
llm=ChatOpenAI(openai_api_key=KEY, model_name="gpt-4", temperature=0.5)   

In [42]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2

In [51]:
RESPONSE_JSON = {
	"1": {
		"mcq": "multiple choice questions",
		"options": {
			"a": "choice here",
			"b": "choice here",
			"c": "choice here",
			"d": "choice here",
		},
        "correct": "correct answer",
	},
	"2": {
		"mcq": "multiple choice questions",
		"options": {
			"a": "choice here",
			"b": "choice here",
			"c": "choice here",
			"d": "choice here",
		},
        "correct": "correct answer",
	},
	"3": {
		"mcq": "multiple choice questions",
		"options": {
			"a": "choice here",
			"b": "choice here",
			"c": "choice here",
			"d": "choice here",
		},
        "correct": "correct answer",
	},
}

In [47]:
TEMPLATE = """
TEXT: {text}
You are an expert mcq maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone}. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well. 
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [49]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [50]:
quiz_chain = LLMChain(
    llm=llm,
    prompt=quiz_generation_prompt, 
    output_key="quiz",
    verbose=True
)

In [52]:
TEMPLATE2 = """
You are an expert english grammarian and writer. Given a multiple choice quiz for {subject} students. \
You need to evaluate the complexity  of the question and give a complete analysis of the quiz. Only use \
at max 50 words for complexity if the quiz is not as per cognitive and analytical abilities of the students, \
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the students.
Quiz_MCQ's:
{quiz}

Check from an expert English Writer of the above quiz.
"""

In [53]:
quiz_evaluation_prompt = PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE2)

In [54]:
review_chain = LLMChain(
    llm=llm,
    prompt=quiz_evaluation_prompt,
    output_key="review",
    verbose=True
)

In [55]:
generate_evaluate_chain = SequentialChain( 
    chains=[quiz_chain, review_chain],
    input_variables=["text", "number", "subject", "tone", "response_json"],
    output_variables=["quiz", "review"],
    verbose=True
)


In [ ]:
file_path = r"D:\mcqgen\data.txt"